# Prática 2 — Fine-Tuning em CIFAR-10
**ENG4502 — Introdução à Ciência de Dados · PUC-Rio**

Neste notebook vamos:
1. Comparar **Feature Extraction** com **Fine-Tuning parcial** usando ResNet-18
2. Entender o conceito de *discriminative learning rates*
3. Visualizar o ganho de acurácia versus o custo computacional adicional
4. Analisar a matriz de confusão de ambas as abordagens

> **Pré-requisito:** ter concluído o notebook `01_feature_extraction_cifar10.ipynb`

**Referência:** Wang & Chen (2023), *Introduction to Transfer Learning*, §3.2 — Fine-Tuning

## 0. Configurações e imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import time

# Classes do CIFAR-10
CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

# Dispositivo (GPU se disponível, senão CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

## 1. Dataset — mesmo pipeline da Prática 1

In [ ]:
transform = transforms.Compose([
    transforms.Resize(224),           # ResNet-18 espera 224×224
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_ds = datasets.CIFAR10('data', train=True,  download=True, transform=transform)
val_ds   = datasets.CIFAR10('data', train=False, download=True, transform=transform)

# num_workers=0 para compatibilidade com Windows
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0)

print(f'Treino: {len(train_ds)} amostras | Validação: {len(val_ds)} amostras')

## 2. Funções auxiliares de treinamento e avaliação

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Treina o modelo por uma época. Retorna a loss média."""
    model.train()
    total_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, device):
    """Calcula acurácia e retorna todas as predições (para matriz de confusão)."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            _, preds = torch.max(model(inputs), 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return acc, all_preds, all_labels


def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs=5, label=''):
    """Loop de treinamento completo. Retorna histórico de loss e acurácia."""
    history = {'loss': [], 'acc': []}
    for epoch in range(num_epochs):
        t0  = time.time()
        loss = train_epoch(model, train_loader, criterion, optimizer, device)
        acc, _, _ = evaluate(model, val_loader, device)
        elapsed = time.time() - t0
        history['loss'].append(loss)
        history['acc'].append(acc)
        print(f'[{label}] Época {epoch+1}/{num_epochs} '
              f'| Loss: {loss:.4f} | Val Acc: {acc:.4f} | {elapsed:.1f}s')
    return history

## 3. Abordagem A — Feature Extraction (referência)

Repetimos rapidamente o resultado da Prática 1 para ter uma linha de base de comparação.

In [ ]:
# ── Modelo A: Feature Extraction ──────────────────────────────────────────
model_fe = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for p in model_fe.parameters():
    p.requires_grad = False           # congela TUDO

model_fe.fc = nn.Linear(model_fe.fc.in_features, 10)
model_fe = model_fe.to(device)

# Apenas a camada final é treinada
trainable_fe = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
print(f'Parâmetros treináveis (Feature Extraction): {trainable_fe:,}')

optimizer_fe = optim.SGD(model_fe.fc.parameters(), lr=0.01, momentum=0.9)
criterion    = nn.CrossEntropyLoss()

print('\n=== Treinamento: Feature Extraction ===')
history_fe = train_model(model_fe, train_loader, val_loader,
                          criterion, optimizer_fe, num_epochs=5, label='FE')

## 4. Abordagem B — Fine-Tuning parcial (layer4 + fc)

Descongelamos apenas o **último bloco residual** (`layer4`) do ResNet-18,
além da camada final. Usamos **discriminative learning rates**: taxa menor
para as camadas descongeladas (preserva features já aprendidas) e taxa
maior para a camada final.

In [ ]:
# ── Modelo B: Fine-Tuning parcial ──────────────────────────────────────────
model_ft = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 1. Congela tudo
for p in model_ft.parameters():
    p.requires_grad = False

# 2. Descongela layer4 (último bloco residual)
for p in model_ft.layer4.parameters():
    p.requires_grad = True   # ← diferença em relação a Feature Extraction

# 3. Substitui camada final
model_ft.fc = nn.Linear(model_ft.fc.in_features, 10)
model_ft = model_ft.to(device)

trainable_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f'Parâmetros treináveis (Fine-Tuning): {trainable_ft:,}')

# 4. Discriminative Learning Rates
#    layer4 → lr baixo (não degradar features já aprendidas)
#    fc     → lr alto  (aprender as novas classes)
optimizer_ft = optim.SGD([
    {'params': model_ft.layer4.parameters(), 'lr': 1e-4},
    {'params': model_ft.fc.parameters(),     'lr': 1e-3},
], momentum=0.9)

print('\n=== Treinamento: Fine-Tuning parcial ===')
history_ft = train_model(model_ft, train_loader, val_loader,
                          criterion, optimizer_ft, num_epochs=5, label='FT')

## 5. Comparação visual: curvas de loss e acurácia

In [ ]:
epochs = range(1, 6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0d1117')

for ax in [ax1, ax2]:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')

# Loss
ax1.plot(epochs, history_fe['loss'], 'o-', color='#58a6ff', label='Feature Extraction')
ax1.plot(epochs, history_ft['loss'], 's-', color='#f78166', label='Fine-Tuning (layer4)')
ax1.set_title('Curva de Loss', color='#e6edf3')
ax1.set_xlabel('Época', color='#8b949e')
ax1.set_ylabel('Loss', color='#8b949e')
ax1.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
ax1.grid(alpha=0.2, color='#30363d')

# Acurácia
ax2.plot(epochs, [a*100 for a in history_fe['acc']], 'o-', color='#3fb950', label='Feature Extraction')
ax2.plot(epochs, [a*100 for a in history_ft['acc']], 's-', color='#e3b341', label='Fine-Tuning (layer4)')
ax2.set_title('Acurácia de Validação (%)', color='#e6edf3')
ax2.set_xlabel('Época', color='#8b949e')
ax2.set_ylabel('Acurácia (%)', color='#8b949e')
ax2.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
ax2.grid(alpha=0.2, color='#30363d')
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('docs/comparison_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"\nResultado final:")
print(f"  Feature Extraction → Acc: {history_fe['acc'][-1]*100:.1f}%")
print(f"  Fine-Tuning (layer4) → Acc: {history_ft['acc'][-1]*100:.1f}%")
print(f"  Ganho: +{(history_ft['acc'][-1] - history_fe['acc'][-1])*100:.1f} pontos percentuais")

## 6. Matriz de confusão comparativa

In [ ]:
_, preds_fe, labels_fe = evaluate(model_fe, val_loader, device)
_, preds_ft, labels_ft = evaluate(model_ft, val_loader, device)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

for ax, preds, labels, title in [
    (ax1, preds_fe, labels_fe, 'Feature Extraction'),
    (ax2, preds_ft, labels_ft, 'Fine-Tuning (layer4)')
]:
    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, color='#e6edf3', pad=12)
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e', labelsize=8)
    ax.set_xlabel('Predito', color='#8b949e')
    ax.set_ylabel('Real', color='#8b949e')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('docs/confusion_matrix_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 7. Discussão e conclusão

### O que observamos?

| Abordagem | Params treináveis | Acurácia (5 épocas) | Tempo estimado |
|-----------|------------------|---------------------|----------------|
| Feature Extraction | ~5.1K | ~75% | ~8 min (CPU) |
| Fine-Tuning (layer4) | ~2.6M | ~82% | ~25 min (CPU) |

### Exercícios para reflexão

1. **Qual seria o impacto de descongelar `layer3` também?** Tente modificar o código e observe a mudança de tempo e acurácia.

2. **Por que usamos learning rates diferentes (`1e-4` para `layer4` e `1e-3` para `fc`)?**  
   Dica: pense no que aconteceria se aplicássemos o mesmo lr alto às camadas já treinadas.

3. **Em que contexto de Engenharia de Produção você escolheria Fine-Tuning em vez de Feature Extraction?**  
   Ex.: detecção de defeitos em peças industriais com 500 imagens rotuladas.

### Referência
Wang, Y., & Chen, X. (2023). *Introduction to Transfer Learning: Algorithms and Practice*. Springer, §3.2.